# 59. DINOv3 Student-Teacher와 Gram Anchoring

            DINO 계열은 같은 이미지에서 만든 서로 다른 augmented view가 비슷한 표현을 갖도록 student와 teacher 네트워크를 학습합니다. DINOv3에서는 긴 학습 동안 patch-level dense feature가 약해지는 문제를 줄이기 위해 Gram anchoring을 도입합니다.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path("Deeplearning") / "Vision 기초" / "8장",
    Path("Vision 기초") / "8장",
]
NOTEBOOK_DIR = next((p for p in candidates if (p / "seg8_utils.py").exists()), Path.cwd())
sys.path.append(str(NOTEBOOK_DIR))

from seg8_utils import *
set_korean_font()
set_seed(7)

DATA_ROOT = ensure_dataset()
RUNS_ROOT = NOTEBOOK_DIR / "runs"


## 59-1. Student-Teacher 흐름


In [ ]:
steps = [
    "한 이미지에서 global crop과 local crop을 만든다.",
    "student는 여러 view를 보고 feature를 예측한다.",
    "teacher는 EMA로 업데이트되며 더 안정적인 target feature를 만든다.",
    "student output이 teacher output과 일관되도록 loss를 계산한다.",
    "label 없이도 object part, texture, layout에 민감한 feature가 형성된다.",
]
for i, step in enumerate(steps, 1):
    print(f"{i}. {step}")


## 59-2. Gram matrix 직관


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
features = rng.normal(size=(64, 32))
features = features / np.linalg.norm(features, axis=1, keepdims=True)
gram = features @ features.T

plt.figure(figsize=(4, 4))
plt.imshow(gram, cmap="viridis")
plt.title("patch feature Gram matrix")
plt.colorbar(fraction=0.046)
plt.axis("off")
plt.show()


## 정리

            Gram anchoring은 patch feature들 사이의 관계 구조를 anchor로 잡는 관점입니다. DINOv3에서 중요한 이유는 image classification처럼 하나의 global vector만 좋은 모델이 아니라, segmentation과 depth처럼 위치별 feature가 중요한 task에서도 backbone을 강하게 유지해야 하기 때문입니다.
